# Prepare GFDL-Named Example Data

Load the multicase example configuration and its intake-ESM catalog, translate daily TAS from CMIP names to GFDL names, and write reusable NetCDF files plus a case-info YAML for the reverse-translation demo.

The source catalog refers to archive NetCDF paths. Run this notebook in an environment where those paths are mounted, or edit the input paths below.

## Load inputs and translator; setup outputs

In [18]:
from pathlib import Path
import copy
import json
import sys
import yaml
import intake

# This notebook is expected to run from the repository or its notebook directory.
start_dir = Path.cwd().resolve()
repo_root = next((path for path in (start_dir, *start_dir.parents) if (path / 'data' / 'fieldlist_CMIP.jsonc').is_file()), None)
if repo_root is None:
    raise FileNotFoundError('Could not locate the repository root. Start Jupyter inside MDTF-diagnostics or set repo_root manually.')
example_dir = repo_root / 'diagnostics' / 'example_multicase'

# Inputs - Case environment file and catalog file [CMIP convention]
case_env_file = example_dir / 'case_info_demo2.yml'
catalog_file = example_dir / 'amip_c96L65_am5f3b1r0_pdclim1850F_combined.json'

# Outputs - Prepared GFDL data files [GFDL convention]
output_dir = example_dir / 'prepared_gfdl_data'
output_case_env_file = example_dir / 'case_info_gfdl_demo.yml'
output_catalog_csv = example_dir / 'prepared_gfdl_data_catalog.csv'
output_catalog_json = example_dir / 'prepared_gfdl_data_catalog.json'

# Change these paths if your local input files are elsewhere.
assert case_env_file.is_file(), f'Case environment file not found: {case_env_file}'
assert catalog_file.is_file(), f'Catalog definition not found: {catalog_file}'
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

with case_env_file.open('r') as stream:
    case_info = yaml.safe_load(stream)

# The checked-in YAML contains a host-specific catalog path; use the sibling copy.
case_info['CATALOG_FILE'] = str(catalog_file)
catalog = intake.open_esm_datastore(str(catalog_file))
print(f'Cases: {list(case_info["CASE_LIST"])}')
print(f'Catalog rows: {len(catalog.df)}')
print(f'Catalog variables: {sorted(catalog.df.variable_id.dropna().unique())}')

Cases: ['c384L65_am5f3b1r0_amip', 'amip_c96L65_am5f3b1r0_pdclim1850F']
Catalog rows: 785
Catalog variables: ['albs', 'ccb', 'cct', 'ci', 'clivi', 'clt', 'clwvi', 'evspsbl', 'hfls', 'hfss', 'hur', 'hur_unmsk', 'hurs', 'hus', 'hus_unmsk', 'huss', 'o3', 'o3_unmsk', 'pr', 'prc', 'prsn', 'prw', 'ps', 'psl', 'rlds', 'rldscs', 'rlus', 'rlut', 'rlutcs', 'rsds', 'rsdscs', 'rsdt', 'rsus', 'rsuscs', 'rsut', 'rsutcs', 'rtmt', 'sci', 'sfcWind', 'ta', 'ta_unmsk', 'tas', 'tasmax', 'tasmin', 'tauu', 'tauv', 'ts', 'ua', 'ua10', 'ua200', 'ua850', 'ua_unmsk', 'uas', 'utendnogw', 'utendogw', 'va', 'va850', 'va_unmsk', 'vas', 'wap', 'wap_unmsk', 'zg', 'zg500', 'zg_unmsk']


## Check the source records

The case YAML supplies each case name and requested source variable. The catalog's `experiment_id` is matched against the case name so the appropriate archive records are selected.

In [19]:
source_frequency = 'daily'
source_convention = 'cmip'
source_variable_name = 'tas' # CMIP convention variable name

target_convention = 'gfdl'

for case_name, case in case_info['CASE_LIST'].items():
    source_var = case['tas_var']
    records = catalog.df[
        (catalog.df['variable_id'] == source_var)
        & (catalog.df['frequency'] == source_frequency)
        & catalog.df['experiment_id'].fillna('').astype(str).apply(lambda value: value in case_name or case_name in value)
    ]
    print(f'{case_name}: {len(records)} matching {source_frequency} {source_var} record(s)')
    if records.empty:
        raise ValueError(f'No matching catalog rows for {case_name}; inspect catalog.df and adjust the matching/filtering rule.')

c384L65_am5f3b1r0_amip: 3 matching daily tas record(s)
amip_c96L65_am5f3b1r0_pdclim1850F: 10 matching daily tas record(s)


## Translate and write case datasets

Each selected case is loaded through intake. `VariableTranslator` maps its variables using their CF metadata; data values and coordinates are preserved, while variable names are changed to the GFDL convention.

In [20]:
from src.translation import VariableTranslator
from src.varlist_util import VarlistEntry
from src.units import Units

translator = VariableTranslator(code_root=str(repo_root))
translator.read_conventions(str(repo_root))

def translate_variable(variable_name, translator, source_convention, target_convention):
    # translate variable based on the source and target conventions
    source_field = translator.get_convention(source_convention).to_CF(variable_name)
    entry = VarlistEntry(
        name=variable_name,
        _parent=None,
        standard_name=source_field.get('standard_name') or '',
        long_name=source_field.get('long_name') or '',
        units=Units(source_field.get('units') or ''),
        realm=source_field.get('realm') or '',
        modifier=source_field.get('modifier') or '',
        coords=[],
        use_exact_name=False,
    )
    translated = translator.translate(entry, target_convention, source_convention)
    if translated is None:
        raise ValueError(
            f'No {source_convention} -> {target_convention} translation found for {variable_name}'
        )
    return translated

translated_variable = translate_variable(
    source_variable_name,
    translator,
    source_convention,
    target_convention,
)
print(
    f'{source_convention.upper()} {source_variable_name} '
    f'-> {target_convention.upper()} {translated_variable.name}'
)

CMIP tas -> GFDL t_ref


In [21]:
import pandas as pd

output_dir.mkdir(parents=True, exist_ok=True)
translated_case_info = copy.deepcopy(case_info)
translated_case_info['CATALOG_FILE'] = str(output_catalog_json)
translated_case_info['CASE_LIST'] = {}
translation_rows = []
output_catalog_rows = []

### Select source records

This stage filters the input catalog to the requested variable, frequency, and case. Inspect the returned rows and paths here before opening data.

In [22]:
def select_case_records(case_name, source_var):
    # Select catalog rows that match the case name and source variable.
    case_mask = (
        (catalog.df['variable_id'] == source_var)
        & (catalog.df['frequency'] == source_frequency)
        & catalog.df['experiment_id'].fillna('').astype(str).apply(
            lambda value: value in case_name or case_name in value
        )
    )
    matching_rows = catalog.df.loc[case_mask]
    if matching_rows.empty:
        raise ValueError(f'No catalog rows found for {case_name} / {source_var}')

    matching_experiments = matching_rows['experiment_id'].dropna().unique()
    if len(matching_experiments) != 1:
        raise ValueError(
            f'{case_name}: expected one matching experiment_id, found {list(matching_experiments)}'
        )
    case_catalog = catalog.search(
        variable_id=source_var,
        frequency=source_frequency,
        experiment_id=str(matching_experiments[0]),
    )
    return matching_rows, case_catalog

### Open one case dataset

This stage opens the selected case's catalog group and checks that the requested source variable is present. The dataset remains lazy until its data are read or written.

In [23]:
def open_case_dataset(case_name, source_var, case_catalog):
    # Open the dataset corresponding to the case and source variable.
    datasets = case_catalog.to_dataset_dict(
        xarray_open_kwargs={'decode_times': True, 'use_cftime': True}
    )
    if len(datasets) != 1:
        raise ValueError(
            f'{case_name}: expected one dataset group, found {len(datasets)}'
        )
    dataset_key, dataset = next(iter(datasets.items()))
    if source_var not in dataset.data_vars:
        raise KeyError(
            f'{case_name}: {source_var!r} is missing from {dataset_key!r}; '
            f'available data variables: {list(dataset.data_vars)}'
        )
    return dataset_key, dataset

### Translate and rename the variable

This stage checks the source variable's CF metadata, calls the shared translation helper, and renames the variable in memory. No output file is written yet.

In [24]:
def translate_case_dataset(case_name, source_var, dataset):
    # Translate the case dataset according to the target convention.
    attrs = dataset[source_var].attrs
    standard_name = attrs.get('standard_name')
    units = attrs.get('units')
    if not standard_name or not units:
        raise ValueError(
            f'{case_name}/{source_var}: expected standard_name and units; found {attrs}'
        )

    translated = translate_variable(
        source_var,
        translator,
        source_convention,
        target_convention,
    )
    target_var = translated.name
    if target_var != source_var and target_var in dataset.variables:
        raise ValueError(
            f'{case_name}: cannot rename {source_var!r} to {target_var!r}; '
            'the target name already exists'
        )
    converted = dataset.rename({source_var: target_var}) if target_var != source_var else dataset
    return converted, target_var, standard_name, units, attrs.get('long_name', '')

### Write one converted dataset

This stage writes the renamed dataset and prepares the matching case-info and catalog rows. Keeping file output separate makes write-time I/O errors easier to distinguish from translation errors.

In [25]:
def write_case_dataset(case_name, case, matching_rows, converted, target_var, standard_name, units, long_name):
    # Write the translated dataset to a NetCDF file.
    output_file = output_dir / f'{case_name}.{target_var}.nc'
    converted.to_netcdf(output_file)

    translated_case = copy.deepcopy(case)
    translated_case['TAS_FILE'] = str(output_file)
    translated_case['tas_var'] = target_var
    translated_case['convention'] = 'GFDL'

    catalog_row = matching_rows.iloc[0].to_dict()
    catalog_row.update({
        'variable_id': target_var,
        'path': str(output_file),
        'standard_name': standard_name,
        'long_name': long_name,
        'units': units,
    })
    translation_row = {
        'case': case_name,
        'source_variable': case['tas_var'],
        'target_variable': target_var,
        'standard_name': standard_name,
        'units': units,
        'file': str(output_file),
    }
    return translated_case, catalog_row, translation_row

### Run conversion for all cases

The loop coordinates the previous stages once per case. If a stage fails, its helper name and current case identify where to inspect.

In [26]:
for case_name, case in case_info['CASE_LIST'].items():
    source_var = case['tas_var']
    matching_rows, case_catalog = select_case_records(case_name, source_var)
    dataset_key, dataset = open_case_dataset(case_name, source_var, case_catalog)
    converted, target_var, standard_name, units, long_name = translate_case_dataset(
        case_name,
        source_var,
        dataset,
    )
    translated_case, catalog_row, translation_row = write_case_dataset(
        case_name,
        case,
        matching_rows,
        converted,
        target_var,
        standard_name,
        units,
        long_name,
    )
    translated_case_info['CASE_LIST'][case_name] = translated_case
    output_catalog_rows.append(catalog_row)
    translation_rows.append(translation_row)
    print(f'{case_name}: {source_var} -> {target_var}; wrote {translation_row["file"]}')
    dataset.close()


--> The keys in the returned dictionary of datasets are constructed as follows:
	'experiment_id.frequency.modeling_realm.variable_id'


<div><progress max="1" value="1"></progress> 100.00% [1/1 00:00&lt;00:00]</div>

/work/Janice.Kim/mdtf_main/conda_env_main/_MDTF_base/lib/python3.12/site-packages/intake_esm/source.py:82: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_dataset(url, **xarray_open_kwargs)
/work/Janice.Kim/mdtf_main/conda_env_main/_MDTF_base/lib/python3.12/site-packages/intake_esm/source.py:82: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_dataset(url, **xarray_open_kwargs)
/work/Janice.Kim/mdtf_main/conda_env_main/_MDTF_base/lib/python3.12/site-packages/intake_esm/source.py:82: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCo

c384L65_am5f3b1r0_amip: tas -> t_ref; wrote /work5/j3k/mdtf_main/MDTF-diagnostics/diagnostics/example_multicase/prepared_gfdl_data/c384L65_am5f3b1r0_amip.t_ref.nc

--> The keys in the returned dictionary of datasets are constructed as follows:
	'experiment_id.frequency.modeling_realm.variable_id'


<div><progress max="1" value="1"></progress> 100.00% [1/1 00:00&lt;00:00]</div>

/work/Janice.Kim/mdtf_main/conda_env_main/_MDTF_base/lib/python3.12/site-packages/intake_esm/source.py:82: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_dataset(url, **xarray_open_kwargs)
/work/Janice.Kim/mdtf_main/conda_env_main/_MDTF_base/lib/python3.12/site-packages/intake_esm/source.py:82: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_dataset(url, **xarray_open_kwargs)
/work/Janice.Kim/mdtf_main/conda_env_main/_MDTF_base/lib/python3.12/site-packages/intake_esm/source.py:82: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCo

amip_c96L65_am5f3b1r0_pdclim1850F: tas -> t_ref; wrote /work5/j3k/mdtf_main/MDTF-diagnostics/diagnostics/example_multicase/prepared_gfdl_data/amip_c96L65_am5f3b1r0_pdclim1850F.t_ref.nc


### Compare CMIP input with GFDL output

For each case, compare dimensions and coordinates, then compare a small slice of values exactly. Set `sample_size` to a larger value to inspect a larger leading slice; this checks the rename-only conversion without loading every full-resolution value.

In [27]:
import numpy as np
import xarray as xr

sample_size = 2
comparison_rows = []
catalog = intake.open_esm_datastore(str(catalog_file))

for case_name, case in case_info['CASE_LIST'].items():
    source_var = case['tas_var']
    matching_rows, case_catalog = select_case_records(case_name, source_var)
    _, cmip_dataset = open_case_dataset(case_name, source_var, case_catalog)
    target_var = translated_case_info['CASE_LIST'][case_name]['tas_var']
    gfdl_file = Path(translated_case_info['CASE_LIST'][case_name]['TAS_FILE'])

    try:
        with xr.open_dataset(gfdl_file, decode_times=True, use_cftime=True) as gfdl_dataset:
            cmip_array = cmip_dataset[source_var]
            gfdl_array = gfdl_dataset[target_var].rename(source_var)

            if cmip_array.dims != gfdl_array.dims or cmip_array.sizes != gfdl_array.sizes:
                raise AssertionError(
                    f'{case_name}: dimensions differ: '
                    f'CMIP={dict(cmip_array.sizes)}, GFDL={dict(gfdl_array.sizes)}'
                )

            sample_indexers = {
                dim: slice(0, min(sample_size, cmip_array.sizes[dim]))
                for dim in cmip_array.dims
            }
            cmip_sample = cmip_array.isel(sample_indexers).compute()
            gfdl_sample = gfdl_array.isel(sample_indexers).compute()
            xr.testing.assert_equal(cmip_sample, gfdl_sample)

            sample_difference = np.abs(cmip_sample.values - gfdl_sample.values)
            max_abs_difference = float(np.nanmax(sample_difference)) if sample_difference.size else 0.0
            comparison_rows.append({
                'case': case_name,
                'CMIP variable': source_var,
                'GFDL variable': target_var,
                'dimensions_match': True,
                'sample_coordinates_and_values_match': True,
                'sample_shape': cmip_sample.shape,
                'sample_max_abs_difference': max_abs_difference,
            })
    finally:
        cmip_dataset.close()

catalog.close()
comparison_results = pd.DataFrame(comparison_rows)
comparison_results


--> The keys in the returned dictionary of datasets are constructed as follows:
	'experiment_id.frequency.modeling_realm.variable_id'


<div><progress max="1" value="1"></progress> 100.00% [1/1 00:00&lt;00:00]</div>

/work/Janice.Kim/mdtf_main/conda_env_main/_MDTF_base/lib/python3.12/site-packages/intake_esm/source.py:82: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_dataset(url, **xarray_open_kwargs)
/work/Janice.Kim/mdtf_main/conda_env_main/_MDTF_base/lib/python3.12/site-packages/intake_esm/source.py:82: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_dataset(url, **xarray_open_kwargs)
/work/Janice.Kim/mdtf_main/conda_env_main/_MDTF_base/lib/python3.12/site-packages/intake_esm/source.py:82: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCo


--> The keys in the returned dictionary of datasets are constructed as follows:
	'experiment_id.frequency.modeling_realm.variable_id'


<div><progress max="1" value="1"></progress> 100.00% [1/1 00:00&lt;00:00]</div>

/work/Janice.Kim/mdtf_main/conda_env_main/_MDTF_base/lib/python3.12/site-packages/intake_esm/source.py:82: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_dataset(url, **xarray_open_kwargs)
/work/Janice.Kim/mdtf_main/conda_env_main/_MDTF_base/lib/python3.12/site-packages/intake_esm/source.py:82: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds = xr.open_dataset(url, **xarray_open_kwargs)
/work/Janice.Kim/mdtf_main/conda_env_main/_MDTF_base/lib/python3.12/site-packages/intake_esm/source.py:82: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCo

,case,CMIP variable,GFDL variable,dimensions_match,sample_coordinates_and_values_match,sample_shape,sample_max_abs_difference
0,c384L65_am5f3b1r0_amip,tas,t_ref,True,True,"(2, 2, 2)",0.0
1,amip_c96L65_am5f3b1r0_pdclim1850F,tas,t_ref,True,True,"(2, 2, 2)",0.0


### Clean output catalog metadata

The source catalog's `modeling_realm` describes the `atmos_cmip` stream. Normalize the prepared catalog's realm metadata to `atmos` before writing it. 

**TODO: Find out if this is an appropriate thing to do. I'm not sure why it was labeled 'atmos_cmip'  instead of 'atmos' in the first place.**

In [28]:
output_catalog_df = pd.DataFrame(output_catalog_rows)
if 'modeling_realm' not in output_catalog_df.columns:
    raise KeyError('Expected modeling_realm column in the generated catalog rows')

print('modeling_realm values before cleanup:')
display(output_catalog_df['modeling_realm'].value_counts(dropna=False))
output_catalog_df['modeling_realm'] = 'atmos'
print('modeling_realm values after cleanup:')
display(output_catalog_df['modeling_realm'].value_counts(dropna=False))

modeling_realm values before cleanup:


modeling_realm
atmos_cmip    2
Name: count, dtype: int64

modeling_realm values after cleanup:


modeling_realm
atmos    2
Name: count, dtype: int64

### Normalize GFDL case-info variable keys

The input configuration uses CMIP-oriented `TAS_FILE` and `tas_var` names. Rename those output fields using each translated GFDL variable name before saving the case-info YAML. 

**TODO: Again, find out if this is an appropriate thing to do. I'm not sure if it makes sense to create a comparable 'T_REF_FILE' for 'TAS_FILE', etc.**

In [29]:
target_variables = {
    row['case']: row['target_variable']
    for row in translation_rows
}

for case_name, case_config in translated_case_info['CASE_LIST'].items():
    target_var = target_variables[case_name]
    target_prefix = target_var.upper()

    if 'TAS_FILE' in case_config:
        case_config[f'{target_prefix}_FILE'] = case_config.pop('TAS_FILE')
    if 'tas_var' in case_config:
        case_config[f'{target_var.lower()}_var'] = case_config.pop('tas_var')
    if 'TAS_ASSOC_FILES' in case_config:
        case_config[f'{target_prefix}_ASSOC_FILES'] = case_config.pop('TAS_ASSOC_FILES')

print('Prepared case-info variable keys:')
for case_name, case_config in translated_case_info['CASE_LIST'].items():
    print(case_name, [key for key in case_config if key.endswith('_FILE') or key.endswith('_var') or key.endswith('_ASSOC_FILES')])

Prepared case-info variable keys:
c384L65_am5f3b1r0_amip ['T_REF_FILE', 't_ref_var', 'T_REF_ASSOC_FILES']
amip_c96L65_am5f3b1r0_pdclim1850F ['T_REF_FILE', 't_ref_var', 'T_REF_ASSOC_FILES']


### Save catalog and case configuration

After every case has converted successfully, write the GFDL catalog files and case environment YAML.

In [30]:
with catalog_file.open('r') as stream:
    output_catalog_definition = json.load(stream)
output_catalog_definition['catalog_file'] = str(output_catalog_csv)
with output_catalog_json.open('w') as stream:
    json.dump(output_catalog_definition, stream, indent=2)
output_catalog_df.to_csv(output_catalog_csv, index=False)

with output_case_env_file.open('w') as stream:
    yaml.safe_dump(translated_case_info, stream, sort_keys=False)
catalog.close()
print(f'Wrote GFDL catalog: {output_catalog_json}')
print(f'Wrote GFDL case info: {output_case_env_file}')

Wrote GFDL catalog: /work5/j3k/mdtf_main/MDTF-diagnostics/diagnostics/example_multicase/prepared_gfdl_data_catalog.json
Wrote GFDL case info: /work5/j3k/mdtf_main/MDTF-diagnostics/diagnostics/example_multicase/case_info_gfdl_demo.yml


## Review generated files

After the conversion cell finishes, this table summarizes each source-to-target variable rename and the NetCDF path written for the downstream example.

In [31]:
import pandas as pd

pd.DataFrame(translation_rows)

,case,source_variable,target_variable,standard_name,units,file
0,c384L65_am5f3b1r0_amip,tas,t_ref,air_temperature,K,/work5/j3k/mdtf_main/MDTF-diagnostics/diagnost...
1,amip_c96L65_am5f3b1r0_pdclim1850F,tas,t_ref,air_temperature,K,/work5/j3k/mdtf_main/MDTF-diagnostics/diagnost...


### Check generated files

Confirm that every file recorded by the conversion stage exists and has nonzero size before comparing its contents.

In [32]:
from pathlib import Path

file_check_rows = []
for row in translation_rows:
    output_path = Path(row['file'])
    file_check_rows.append({
        'case': row['case'],
        'gfdl_variable': row['target_variable'],
        'exists': output_path.is_file(),
        'size_mb': round(output_path.stat().st_size / (1024 ** 2), 2) if output_path.is_file() else None,
        'file': str(output_path),
    })

file_check = pd.DataFrame(file_check_rows)
file_check

,case,gfdl_variable,exists,size_mb,file
0,c384L65_am5f3b1r0_amip,t_ref,True,3467.93,/work5/j3k/mdtf_main/MDTF-diagnostics/diagnost...
1,amip_c96L65_am5f3b1r0_pdclim1850F,t_ref,True,722.01,/work5/j3k/mdtf_main/MDTF-diagnostics/diagnost...
